# 02 — Text to Vectors

Runnable companion to **notes 09–12**: one-hot encoding, bag of words, n-grams and TF-IDF.

Every technique is computed **twice** — once by hand with NumPy so you can see the arithmetic,
once with scikit-learn so you can see it agree (or, in TF-IDF's case, see exactly *why* it
disagrees).

| Section | Note |
|---|---|
| 1. The running example | — |
| 2. One-hot encoding + its 4 failures | [09](../notes/09-One-Hot-Encoding.md) |
| 3. Bag of words | [10](../notes/10-Bag-of-Words.md) |
| 4. N-grams | [11](../notes/11-N-Grams.md) |
| 5. TF-IDF by hand | [12](../notes/12-TF-IDF.md) |
| 6. TF-IDF in sklearn, and why the numbers differ | 12 |
| 7. Side-by-side comparison | — |

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 120)
np.set_printoptions(precision=4, suppress=True)

## 1. The running example

Three tiny documents, used for every technique in this notebook so the outputs are directly
comparable.

In [ ]:
corpus_raw = ["the food is good",
              "the food is bad",
              "pizza is amazing"]

for i, d in enumerate(corpus_raw, 1):
    print(f"D{i}: {d}")

vocab = sorted({w for d in corpus_raw for w in d.split()})
print("\nvocabulary (V =", len(vocab), "):", vocab)

## 2. One-hot encoding  ([note 09](../notes/09-One-Hot-Encoding.md))

Each **word** becomes a V-length vector with a single 1.

In [ ]:
V   = len(vocab)
idx = {w: i for i, w in enumerate(vocab)}

def onehot_word(word):
    v = np.zeros(V, dtype=int)
    v[idx[word]] = 1
    return v

pd.DataFrame([onehot_word(w) for w in vocab], index=vocab, columns=vocab)

### 2.1 A document is a *stack* of word vectors — and the shapes disagree

In [ ]:
def onehot_doc(doc):
    return np.array([onehot_word(w) for w in doc.split()])

for i, d in enumerate(corpus_raw, 1):
    print(f"D{i}: {d!r:22} shape = {onehot_doc(d).shape}")

```
   D1 -> 4 x 7
   D2 -> 4 x 7
   D3 -> 3 x 7    <- DIFFERENT
```

**This is disadvantage #2 and it is fatal.** Every ML algorithm needs a fixed number of
features per sample. You cannot `fit()` on this.

In [ ]:
print("D1 = 'the food is good'")
print(pd.DataFrame(onehot_doc(corpus_raw[0]),
                   index=corpus_raw[0].split(), columns=vocab))

### 2.2 Disadvantage #1 — sparsity

In [ ]:
m = onehot_doc(corpus_raw[0])
print(f"entries: {m.size}   non-zero: {m.sum()}   zeros: {100*(1 - m.sum()/m.size):.1f}%")

# now at a realistic vocabulary size
V_real, sent_len = 50_000, 15
print(f"\nwith V={V_real:,} and a {sent_len}-word sentence:")
print(f"  entries : {V_real*sent_len:,}")
print(f"  non-zero: {sent_len}")
print(f"  zeros   : {100*(1 - sent_len/(V_real*sent_len)):.4f}%")

### 2.3 Disadvantage #3 — no semantics: every word is equidistant

In [ ]:
from itertools import combinations

trio = ['food', 'pizza', 'amazing']
vecs = {w: onehot_word(w) for w in trio}

for a, b in combinations(trio, 2):
    d = np.linalg.norm(vecs[a] - vecs[b])
    print(f"distance({a:8}, {b:8}) = {d:.4f}")

**All identical.** One-hot cannot express that `food` and `pizza` are related while
`amazing` is a different kind of word — every pair of distinct one-hot vectors is orthogonal
by construction.

### 2.4 Disadvantage #4 — out of vocabulary

In [ ]:
test = "burger is bad"
for w in test.split():
    print(f"{w:8} -> {'OK, column ' + str(idx[w]) if w in idx else 'OUT OF VOCABULARY - no vector exists'}")

## 3. Bag of words  ([note 10](../notes/10-Bag-of-Words.md))

One vector per **document**, of length V. Shapes finally agree.

### 3.1 By hand

In [ ]:
from collections import Counter

def bow_doc(doc, vocabulary):
    c = Counter(doc.split())
    return np.array([c[w] for w in vocabulary])

X_manual = np.array([bow_doc(d, vocab) for d in corpus_raw])
pd.DataFrame(X_manual, index=['D1','D2','D3'], columns=vocab)

In [ ]:
print("shape:", X_manual.shape, " <- every document is now exactly", len(vocab), "numbers")

### 3.2 With `CountVectorizer`

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
X  = cv.fit_transform(corpus_raw).toarray()

print("features:", cv.get_feature_names_out())
pd.DataFrame(X, index=['D1','D2','D3'], columns=cv.get_feature_names_out())

In [ ]:
print("matches the manual version?", np.array_equal(X, X_manual))

### 3.3 `vocabulary_` maps word -> COLUMN INDEX (not a count)

In [ ]:
print(cv.vocabulary_)
print()
print("'good' lives in column", cv.vocabulary_['good'])
print("columns are sorted ALPHABETICALLY, not by frequency")

### 3.4 Binary vs normal bag of words

In [ ]:
repeated = ["good boy good", "good girl", "boy girl good"]

normal = CountVectorizer()
binary = CountVectorizer(binary=True)

Xn = normal.fit_transform(repeated).toarray()
Xb = binary.fit_transform(repeated).toarray()

print("NORMAL (counts):")
print(pd.DataFrame(Xn, columns=normal.get_feature_names_out()))
print("\nBINARY (presence):")
print(pd.DataFrame(Xb, columns=binary.get_feature_names_out()))
print("\nmax value  normal:", Xn.max(), "  binary:", Xb.max())

Use `binary=True` for **short** documents (SMS, tweets), where a repeat rarely means
"twice as relevant".

### 3.5 ❌ Word order is destroyed — the defining limitation

In [ ]:
pair = ["the man bit the dog", "the dog bit the man"]
cv2  = CountVectorizer()
Xp   = cv2.fit_transform(pair).toarray()

print(pd.DataFrame(Xp, index=['man bites dog','dog bites man'],
                   columns=cv2.get_feature_names_out()))
print("\nvectors identical?", np.array_equal(Xp[0], Xp[1]), " <- opposite meanings, same vector")

### 3.6 ❌ And the `not` disaster

In [ ]:
neg = ["the food is good", "the food is not good"]
cv3 = CountVectorizer()
Xneg = cv3.fit_transform(neg).toarray()

print(pd.DataFrame(Xneg, index=['positive','negative'], columns=cv3.get_feature_names_out()))

cos = np.dot(Xneg[0], Xneg[1]) / (np.linalg.norm(Xneg[0]) * np.linalg.norm(Xneg[1]))
print(f"\ncosine similarity = {cos:.4f}  <- ~89% 'similar', but they are OPPOSITES")

### 3.7 `max_features`, `min_df`, `max_df`

In [ ]:
bigger = corpus_raw + [
    "the pizza is good", "the burger is bad", "good good good food",
    "amazing pizza amazing service", "terrible awful dreadful experience",
]

for mf in [None, 5, 3]:
    v = CountVectorizer(max_features=mf)
    v.fit(bigger)
    print(f"max_features={str(mf):5} -> {len(v.get_feature_names_out()):2} features: "
          f"{sorted(v.get_feature_names_out())}")

In [ ]:
v = CountVectorizer(min_df=2)       # word must appear in >= 2 documents
v.fit(bigger)
print("min_df=2 keeps only words in 2+ documents:", sorted(v.get_feature_names_out()))

v = CountVectorizer(max_df=0.5)    # and in <= 50% of documents
v.fit(bigger)
print("max_df=0.5 drops corpus-wide filler  :", sorted(v.get_feature_names_out()))

## 4. N-grams  ([note 11](../notes/11-N-Grams.md))

The fix for §3.6.

In [ ]:
for rng in [(1,1), (1,2), (2,2), (1,3)]:
    v = CountVectorizer(ngram_range=rng)
    X = v.fit_transform(neg).toarray()
    print(f"ngram_range={rng}  ->  {len(v.get_feature_names_out())} features")
    print("   ", list(v.get_feature_names_out()))
    print("   ", X.tolist())
    print()

### 4.1 Watch the vectors separate

In [ ]:
for rng in [(1,1), (1,2), (2,2)]:
    v = CountVectorizer(ngram_range=rng)
    X = v.fit_transform(neg).toarray()
    cos = np.dot(X[0], X[1]) / (np.linalg.norm(X[0]) * np.linalg.norm(X[1]))
    n_diff = int((X[0] != X[1]).sum())
    print(f"ngram_range={rng}:  cosine={cos:.4f}   differing positions={n_diff}/{X.shape[1]}")

```
   (1,1)  ->  ~0.89 similar   1 position differs   <- model cannot separate them
   (1,2)  ->  much lower      more differ
   (2,2)  ->  0.0             share nothing        <- maximum separation
```

### 4.2 The cost: feature explosion

In [ ]:
longer = ["the quick brown fox jumps over the lazy dog"] * 1 + bigger

for rng in [(1,1), (1,2), (1,3), (1,4)]:
    v = CountVectorizer(ngram_range=rng)
    v.fit(longer)
    print(f"ngram_range={rng}  ->  {len(v.get_feature_names_out()):4} features")

**Always pair `ngram_range` with `max_features` or `min_df`**, or you trade the negation
problem for an overfitting problem.

### 4.3 What n-grams cannot fix — long-distance negation

In [ ]:
far = ["I would not, given everything I have read about it, call this good",
       "I would, given everything I have read about it, call this good"]

v = CountVectorizer(ngram_range=(1,2))
X = v.fit_transform(far).toarray()
cos = np.dot(X[0], X[1]) / (np.linalg.norm(X[0]) * np.linalg.norm(X[1]))
print(f"cosine with bigrams = {cos:.4f}  <- still nearly identical")
print("'not' and 'good' are 9 words apart; no practical n-gram reaches that far.")
print("Fixing this needs an RNN/LSTM or a Transformer.")

## 5. TF-IDF by hand  ([note 12](../notes/12-TF-IDF.md))

Using the cleaned three-sentence corpus from the note (stopwords already removed).

In [ ]:
docs = ["good boy", "good girl", "boy girl good"]
V2   = ['good', 'boy', 'girl']
N    = len(docs)

tokens = [d.split() for d in docs]

### 5.1 Term Frequency = (count in doc) / (words in doc)

In [ ]:
tf = np.array([[t.count(w) / len(t) for w in V2] for t in tokens])
pd.DataFrame(tf, index=['S1','S2','S3'], columns=V2).round(4)

### 5.2 Inverse Document Frequency = ln(N / df)

In [ ]:
df_counts = np.array([sum(w in t for t in tokens) for w in V2])
idf       = np.log(N / df_counts)

pd.DataFrame({'appears in (df)': df_counts,
              'N/df':            N / df_counts,
              'IDF = ln(N/df)':  idf.round(4)}, index=V2)

**`good` appears in all 3 documents, so `ln(3/3) = ln(1) = 0`.** It is about to be
multiplied out of existence — TF-IDF discovered it was uninformative without any stopword
list.

### 5.3 TF x IDF

In [ ]:
tfidf_manual = tf * idf
pd.DataFrame(tfidf_manual, index=['S1','S2','S3'], columns=V2).round(4)

```
              good     boy      girl
   S1    [    0.0    0.2027    0.0    ]   "good boy"      -> the signal is BOY
   S2    [    0.0    0.0       0.2027 ]   "good girl"     -> the signal is GIRL
   S3    [    0.0    0.1352    0.1352 ]   "boy girl good" -> both, but weaker (longer doc)
```

Compare with bag of words, where all three rows were just 1s and 0s with every present word
scoring an identical 1.

In [ ]:
cv_cmp = CountVectorizer()
print("BAG OF WORDS:")
print(pd.DataFrame(cv_cmp.fit_transform(docs).toarray(),
                   index=['S1','S2','S3'], columns=cv_cmp.get_feature_names_out()))

## 6. TF-IDF in sklearn — and why the numbers differ

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tv = TfidfVectorizer()
X_sk = tv.fit_transform(docs).toarray()

pd.DataFrame(X_sk, index=['S1','S2','S3'], columns=tv.get_feature_names_out()).round(4)

**"That's not what I calculated!"** Correct. sklearn differs in three ways.

### 6.1 Difference (a) — smoothed IDF: `ln((1+N)/(1+df)) + 1`

In [ ]:
print("sklearn idf_ :", dict(zip(tv.get_feature_names_out(), tv.idf_.round(4))))

manual_smooth = {w: round(np.log((1+N)/(1+d)) + 1, 4) for w, d in zip(V2, df_counts)}
print("hand-computed:", manual_smooth)
print()
print("note: 'good' gets IDF = 1.0, NOT 0 -- the trailing +1 means IDF is never zero,")
print("so a universal word is down-weighted rather than annihilated.")

### 6.2 Difference (b) — TF is the raw **count**, not count/length
### 6.3 Difference (c) — every row is **L2-normalised** to unit length

In [ ]:
print("row norms in sklearn's output:", np.linalg.norm(X_sk, axis=1).round(6))
print("(all exactly 1.0 -- that is what norm='l2' does)")

In [ ]:
# reproduce sklearn's row for S1 = "good boy", step by step
order  = list(tv.get_feature_names_out())     # ['boy', 'girl', 'good'] -- alphabetical
counts = np.array([docs[0].split().count(w) for w in order])   # RAW counts, sklearn's order

raw = counts * tv.idf_
l2  = raw / np.linalg.norm(raw)

print("column order      :", order)
print("raw counts        :", counts)

print("raw  (count x idf):", raw.round(4))
print("after L2 normalise:", l2.round(4))
print("sklearn row 1     :", X_sk[0].round(4))
print("match?", np.allclose(l2, X_sk[0]))

### 6.4 Turning the differences off to recover the textbook numbers

In [ ]:
tv_raw = TfidfVectorizer(norm=None, smooth_idf=False)
X_raw  = tv_raw.fit_transform(docs).toarray()

print("idf with smoothing off:", dict(zip(tv_raw.get_feature_names_out(), tv_raw.idf_.round(4))))
print()
print(pd.DataFrame(X_raw, index=['S1','S2','S3'],
                   columns=tv_raw.get_feature_names_out()).round(4))
print("\n(still raw counts rather than count/length, so not identical to the hand version --")
print(" but 'good' now carries IDF = 1.0 = ln(1)+1, the textbook shape.)")

**Learn the textbook formula for exams, use sklearn's for code, and be able to explain
the difference** — it is a common interview follow-up.

### 6.5 ❌ TF-IDF still has no idea that synonyms exist

In [ ]:
syn = ["great movie", "excellent film"]
tv_s = TfidfVectorizer()
Xs   = tv_s.fit_transform(syn).toarray()

print("features:", tv_s.get_feature_names_out())
print(Xs)
print("\ncosine similarity =",
      float(np.dot(Xs[0], Xs[1]) / (np.linalg.norm(Xs[0]) * np.linalg.norm(Xs[1]))))
print("ZERO. No shared columns -> treated as completely unrelated documents.")
print("Word2Vec (notebook 03) is what fixes this.")

## 7. Side-by-side comparison

In [ ]:
sentences = ["the food is good", "the food is not good", "the pizza is amazing"]

results = {}
results['Bag of Words']    = CountVectorizer()
results['BoW binary']      = CountVectorizer(binary=True)
results['BoW (1,2)']       = CountVectorizer(ngram_range=(1,2))
results['TF-IDF']          = TfidfVectorizer()
results['TF-IDF (1,2)']    = TfidfVectorizer(ngram_range=(1,2))

for name, vec in results.items():
    X = vec.fit_transform(sentences).toarray()
    cos = np.dot(X[0], X[1]) / (np.linalg.norm(X[0]) * np.linalg.norm(X[1]))
    print(f"{name:15} features={X.shape[1]:3}   "
          f"cos('good' vs 'not good')={cos:.4f}")

Lower cosine between sentence 1 and sentence 2 is **better** here — those two sentences
are opposites, so a good representation should keep them far apart. Watch n-grams do the work.

### 7.1 The scorecard

In [ ]:
scorecard = pd.DataFrame({
    'One-hot': ['No','No','No','No','No','No'],
    'BoW':     ['No','YES','No','Barely','No','No'],
    'TF-IDF':  ['No','YES','YES','Barely','No','with n-grams'],
    'Word2Vec':['YES','YES','Implicit','YES','Mostly','No'],
}, index=['Dense (not sparse)','Fixed-size input','Word importance',
          'Semantic meaning','Handles OOV','Word order'])

scorecard

---

## Exercises

1. Build the one-hot matrix for `"pizza is amazing"` by hand and confirm it is 3 x 7.
2. Show a sentence pair that BoW makes identical but `ngram_range=(1,2)` separates.
3. Compute TF-IDF by hand for a 4-document corpus of your own, then verify with
   `TfidfVectorizer(norm=None, smooth_idf=False)`.
4. On a real dataset, plot accuracy against `max_features` in
   `[100, 500, 1000, 2500, 5000, 10000]`. Where does it plateau?
5. Compare `TfidfVectorizer(sublinear_tf=True)` with the default on long documents.

**Next notebook:** [`03-word2vec-gensim.ipynb`](03-word2vec-gensim.ipynb) — dense vectors that
actually know what words mean.